# Linear Attention Replacement — Experiments 1, 2, 3

Follow-up to the first run (linear running state beat attention on val
loss 1.608 vs 1.634 with fewer params, but the naive Python loop was
~10x slower per batch: 109.7ms vs 11.2ms).

- **Exp 1 — the quadratic killer:** scale context T = 128 → 256 → 512 → 1024.
  Attention is O(T²), the linear state is O(T). Watch the gap explode.
- **Exp 2 — parallel scan speed:** loop vs parallel-scan vs attention,
  milliseconds per batch on GPU. The scan should erase the 10x slowdown.
- **Exp 3 — scale up:** ~6M-param models (dim 256, 8 layers). Does the
  linear compression hold up on harder linguistic structure?


## 0. Setup

In [ ]:
import torch
print("torch", torch.__version__)
print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
device = "cuda" if torch.cuda.is_available() else "cpu"


## 1. Data (same 2.47 MB corpus)

In [ ]:
import os, urllib.request
os.makedirs("data", exist_ok=True)
URLS = {
    "tinyshakespeare.txt": "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    "alice.txt":           "https://www.gutenberg.org/cache/epub/11/pg11.txt",
    "frankenstein.txt":    "https://www.gutenberg.org/cache/epub/84/pg84.txt",
    "pride.txt":           "https://www.gutenberg.org/cache/epub/1342/pg1342.txt",
}
parts = []
for name, url in URLS.items():
    p = f"data/{name}"
    if not os.path.exists(p):
        urllib.request.urlretrieve(url, p)
    with open(p, encoding="utf-8", errors="ignore") as f:
        parts.append(f.read())
text = "\n".join(parts)
chars = sorted(set(text)); vocab = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
n = int(0.95 * len(data)); train_data = data[:n]
print(f"corpus {len(text)/1e6:.2f} MB, vocab {vocab}")

def get_batch(seq_len, batch):
    i = torch.randint(0, len(train_data) - seq_len - 1, (batch,))
    x = torch.stack([train_data[j:j+seq_len] for j in i]).to(device)
    y = torch.stack([train_data[j+1:j+seq_len+1] for j in i]).to(device)
    return x, y


## 2. Models — attention, linear-loop, linear-scan (all inlined, no imports)

In [ ]:
"""
Linear-recurrent replacement for Transformer self-attention.

Experiment: take a standard decoder-only Transformer, rip out the
causal self-attention layer, and replace it with a *moving linear
equation* that compresses the past into a fixed-size running state:

    h_t = d * h_{t-1} + (1 - d) * (B x_t)        (linear recurrence)
    y_t = C h_t                                   (readout)

where d in (0,1)^S is a learned per-channel decay, B/C are learned
projections, and h has FIXED size S independent of sequence length.

This is O(T) time and O(1) memory w.r.t. context length, vs O(T^2)
for attention. The question: how much modeling power do we lose?
"""

import math
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------
# Token mixing layers
# ----------------------------------------------------------------------------

class CausalSelfAttention(nn.Module):
    """Standard multi-head causal self-attention (the baseline)."""
    def __init__(self, dim, n_heads, dropout=0.0):
        super().__init__()
        assert dim % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.out = nn.Linear(dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)                       # (B,T,H,D)
        q, k, v = (t.transpose(1, 2) for t in (q, k, v))  # (B,H,T,D)
        y = F.scaled_dot_product_attention(
            q, k, v, is_causal=True, dropout_p=self.dropout.p if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.out(y)


def parallel_linear_recurrence(u, decay):
    """Parallel associative scan (Hillis-Steele) for h_t = d*h_{t-1} + c_t.

    Reformulates the sequential loop as a tree reduction:
      - each step is an affine map  h -> a*h + b, packed as pair (a, b)
      - pair composition is associative:
            (a1,b1) then (a2,b2)  =  (a2*a1, a2*b1 + b2)
      - a parallel prefix over pairs yields every h_t at once

    log2(T) fully-parallel passes instead of T sequential steps.
    u: (B, T, S) inputs already projected; decay: (S,) in (0, 1).
    Returns h: (B, T, S) — identical math to the for-loop version.
    """
    Bsz, T, S = u.shape
    c = (1.0 - decay) * u
    A = decay.expand(Bsz, T, S).contiguous()   # a-coefficients
    Bh = c.contiguous()                        # b-coefficients (= h after scan)
    idx = torch.arange(T, device=u.device).view(1, T, 1)
    o = 1
    while o < T:
        m = idx >= o
        A_sh = torch.roll(A, o, dims=1)
        B_sh = torch.roll(Bh, o, dims=1)
        # compose segment [t-o, t-1] (pair A_sh,B_sh) then [t] (pair A,Bh):
        #   a_new = a_t * a_{t-o},  b_new = a_t * b_{t-o} + b_t
        A_new = A * A_sh
        B_new = A * B_sh + Bh
        A = torch.where(m, A_new, A)
        Bh = torch.where(m, B_new, Bh)
        o *= 2
    return Bh


class LinearRunningState(nn.Module):
    """Attention replacement: fixed-size running state, linear recurrence.

    h_t = decay * h_{t-1} + (1 - decay) * (B x_t)
    y_t = C h_t

    - decay: learned per-state-channel forget factor in (0, 1)
    - B:     input  projection  dim -> state_dim
    - C:     output projection  state_dim -> dim
    State size is FIXED (does not grow with context length).
    mode="loop": native Python for-loop (sequential, slow on GPU)
    mode="scan": parallel associative scan (log2(T) parallel passes)
    """
    def __init__(self, dim, state_dim=None, dropout=0.0, mode="loop"):
        super().__init__()
        self.mode = mode
        self.state_dim = state_dim or dim
        # init decay ~ U(0.9, 0.999) -> long-ish memory at start
        init = torch.empty(self.state_dim).uniform_(math.log(0.9 / 0.1),
                                                   math.log(0.999 / 0.001))
        self.logit_decay = nn.Parameter(init)
        self.in_proj = nn.Linear(dim, self.state_dim, bias=False)
        self.out_proj = nn.Linear(self.state_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, _ = x.shape
        decay = torch.sigmoid(self.logit_decay)          # (S,) in (0,1)
        u = self.in_proj(x)                              # (B,T,S)
        if self.mode == "scan":
            h_seq = parallel_linear_recurrence(u, decay)  # (B,T,S)
        else:
            keep = 1.0 - decay
            h = torch.zeros(B, self.state_dim, device=x.device, dtype=x.dtype)
            outs = []
            for t in range(T):
                h = decay * h + keep * u[:, t]
                outs.append(h)
            h_seq = torch.stack(outs, dim=1)              # (B,T,S)
        return self.dropout(self.out_proj(h_seq))


# ----------------------------------------------------------------------------
# Blocks & models (identical except for the mixing layer)
# ----------------------------------------------------------------------------

class Block(nn.Module):
    def __init__(self, dim, n_heads, mixer, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)
        self.mix = mixer
        self.mlp = nn.Sequential(
            nn.Linear(dim, 4 * dim), nn.GELU(),
            nn.Linear(4 * dim, dim), nn.Dropout(dropout))

    def forward(self, x):
        x = x + self.mix(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class TinyLM(nn.Module):
    """Decoder-only LM. mixer_fn(dim, n_heads) builds the token-mixing layer."""
    def __init__(self, vocab, dim=128, n_layers=4, n_heads=4,
                 seq_len=128, mixer_fn=None, dropout=0.0):
        super().__init__()
        self.seq_len = seq_len
        self.tok_emb = nn.Embedding(vocab, dim)
        self.pos_emb = nn.Embedding(seq_len, dim)
        self.blocks = nn.ModuleList(
            [Block(dim, n_heads, mixer_fn(dim, n_heads), dropout)
             for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying
        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(
            torch.arange(T, device=idx.device))[None]
        for blk in self.blocks:
            x = blk(x)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new, temp=1.0, top_k=None):
        for _ in range(max_new):
            ctx = idx[:, -self.seq_len:]
            logits, _ = self(ctx)
            logits = logits[:, -1, :] / max(temp, 1e-6)
            if top_k:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx


def count_params(m):
    return sum(p.numel() for p in m.parameters())


def build_models(vocab, dim=128, n_layers=4, n_heads=4, seq_len=128,
                 state_dim=None, dropout=0.0):
    attn = TinyLM(vocab, dim, n_layers, n_heads, seq_len,
                  mixer_fn=lambda d, h: CausalSelfAttention(d, h, dropout),
                  dropout=dropout)
    lin = TinyLM(vocab, dim, n_layers, n_heads, seq_len,
                 mixer_fn=lambda d, h: LinearRunningState(d, state_dim, dropout),
                 dropout=dropout)
    return attn, lin


In [ ]:
# extend the builder: mode="loop" or "scan" for the linear model
def build_all(vocab, dim=128, n_layers=4, n_heads=4, seq_len=128, dropout=0.0):
    attn = TinyLM(vocab, dim, n_layers, n_heads, seq_len,
                  mixer_fn=lambda d, h: CausalSelfAttention(d, h, dropout))
    lin_loop = TinyLM(vocab, dim, n_layers, n_heads, seq_len,
                  mixer_fn=lambda d, h: LinearRunningState(d, None, dropout, mode="loop"))
    lin_scan = TinyLM(vocab, dim, n_layers, n_heads, seq_len,
                  mixer_fn=lambda d, h: LinearRunningState(d, None, dropout, mode="scan"))
    return {"attention": attn, "linear-loop": lin_loop, "linear-scan": lin_scan}

def count_params(m):
    return sum(p.numel() for p in m.parameters())

# correctness gate: scan must equal loop before we trust any timing
torch.manual_seed(0)
_chk = build_all(vocab, dim=64, n_layers=1, n_heads=2, seq_len=64)
_chk = {k: v.to(device) for k, v in _chk.items()}
_chk["linear-loop"].load_state_dict(_chk["linear-scan"].state_dict())
_chk["linear-loop"].eval(); _chk["linear-scan"].eval()
_xb, _yb = get_batch(64, 4)
with torch.no_grad():
    _a = _chk["linear-loop"](_xb)[0]; _b = _chk["linear-scan"](_xb)[0]
print("scan-vs-loop max diff:", float((_a - _b).abs().max()))
del _chk


## Experiment 1 — Context scaling: the quadratic killer

T = 128 → 256 → 512 → 1024. For each: wall time per batch (default
backend, what people actually run) and peak CUDA memory with the math
backend forced (exposes attention's true O(T²) footprint — production
flash attention saves memory but NOT the quadratic FLOPs).


In [ ]:
import time

def bench_time(model, seq_len, batch, iters=20):
    model.train()
    xb, yb = get_batch(seq_len, batch)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
    torch.cuda.synchronize() if device == "cuda" else None
    t0 = time.time()
    for _ in range(iters):
        _, loss = model(xb, yb)
        opt.zero_grad(); loss.backward(); opt.step()
    torch.cuda.synchronize() if device == "cuda" else None
    return (time.time() - t0) / iters * 1000

def bench_mem(model, seq_len, batch):
    if device != "cuda":
        return float("nan")
    model.train()
    xb, yb = get_batch(seq_len, batch)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
    torch.cuda.reset_peak_memory_stats()
    try:
        with torch.backends.cuda.sdp_kernel(enable_flash=False,
                                            enable_mem_efficient=False,
                                            enable_math=True):
            _, loss = model(xb, yb)
            opt.zero_grad(); loss.backward(); opt.step()
        return torch.cuda.max_memory_allocated() / 1e6
    except RuntimeError as e:
        return float("nan") if "out of memory" in str(e).lower() else (_ for _ in ()).throw(e)
    finally:
        torch.cuda.empty_cache()

print(f"{'T':>6} | {'model':<12} | {'ms/batch':>9} | {'peak MB (math bwd)':>18}")
print("-" * 55)
exp1 = {}
for T in [128, 256, 512, 1024]:
    models = build_all(vocab, dim=128, n_layers=2, n_heads=4, seq_len=T)
    exp1[T] = {}
    for name, m in models.items():
        if name == "linear-loop" and T >= 512:
            print(f"{T:>6} | {name:<12} | {'skipped (too slow)':>9} |")
            continue
        m = m.to(device)
        ms = bench_time(m, T, batch=8)
        mb = bench_mem(m, T, batch=8)
        exp1[T][name] = (ms, mb)
        print(f"{T:>6} | {name:<12} | {ms:>9.1f} | {mb:>18.0f}")
    del models
print("EXP1 DONE")


## Experiment 2 — Loop vs parallel scan vs attention (pure speed)

Milliseconds per train batch (forward + backward) on the GPU.
If the scan works, linear-scan should beat attention outright —
erasing the 109.7ms vs 11.2ms deficit from run 1.


In [ ]:
print(f"{'T':>6} | {'model':<12} | {'ms/batch':>9} |")
print("-" * 34)
for T, bs in [(128, 32), (512, 8)]:
    models = build_all(vocab, dim=128, n_layers=4, n_heads=4, seq_len=T)
    for name, m in models.items():
        m = m.to(device)
        ms = bench_time(m, T, batch=bs, iters=30 if T == 128 else 15)
        print(f"{T:>6} | {name:<12} | {ms:>9.1f} |")
    del models
print("EXP2 DONE")


## Experiment 3 — Scale up to ~6M params

dim=256, 8 layers, 8 heads → ~6.3M params each. Train attention vs
linear-**scan** for 1500 steps and compare final val loss: does the
fixed-size compression hold up on harder structure?


In [ ]:
import math

DIM, LAYERS, HEADS, SEQ, BS, STEPS, EVAL = 256, 8, 8, 128, 32, 1500, 250
models = {
    "attention": build_all(vocab, DIM, LAYERS, HEADS, SEQ)["attention"].to(device),
    "linear-scan": build_all(vocab, DIM, LAYERS, HEADS, SEQ)["linear-scan"].to(device),
}
for name, m in models.items():
    print(f"{name}: {count_params(m)/1e6:.2f}M params")
opt = {n: torch.optim.AdamW(m.parameters(), lr=3e-4) for n, m in models.items()}

def val_loss(m):
    m.eval()
    with torch.no_grad():
        return sum(float(m(*get_batch(SEQ, BS))[1]) for _ in range(10)) / 10

for step in range(1, STEPS + 1):
    for name, m in models.items():
        m.train()
        xb, yb = get_batch(SEQ, BS)
        _, loss = m(xb, yb)
        opt[name].zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        opt[name].step()
    if step % EVAL == 0 or step == 1:
        msg = f"step {step:5d}"
        for name, m in models.items():
            vl = val_loss(m)
            msg += f" | {name}: val {vl:.3f} (ppl {math.exp(vl):.1f})"
        print(msg, flush=True)
print("EXP3 DONE")
for name, m in models.items():
    vl = val_loss(m)
    print(f"FINAL {name}: val {vl:.3f} (ppl {math.exp(vl):.1f})")


## How to read the three experiments

- **Exp 1:** attention's ms/batch and peak MB should grow ~4x per doubling
  of T (quadratic); linear-scan should grow ~2x (linear). At T=1024 the
  ratio is the whole argument.
- **Exp 2:** linear-scan ms/batch vs attention ms/batch — the moment the
  10x deficit flips into a win.
- **Exp 3:** final val losses at ~6M params — does the linear state still
  match/beat attention when the modeling gets harder?
